# Experiment 5: Testing of Dead Salmonella Cells

## Overview
Compare spectral signals of heat-killed (dead) vs. live Salmonella cells to distinguish between them using SERS sensor data.

**Protocol Alignment**: This notebook implements Experiment 5 from the Official Testing Protocol for Fiber Optics SERS Sensor (Turkey Rinsate).

## Objective
Compare spectral signals of heat-killed vs. live cells to distinguish between live and dead Salmonella.

## Key Concepts
- **Live vs. dead cell differentiation**: Distinguishing between viable and non-viable bacterial cells
- **Heat-killed samples**: Using heat treatment to create dead cell controls
- **Spectral differences**: Identifying unique signal characteristics for live vs. dead cells
- **Classification**: Machine learning models to classify cell viability status
- **Signal comparison**: Statistical comparison of live vs. dead cell spectra

## Protocol Requirements
1. ⏳ Load and organize SERS data for live and heat-killed Salmonella samples
2. ⏳ Extract spectral features that distinguish live from dead cells
3. ⏳ Compare signal characteristics between live and dead cells
4. ⏳ Develop classification models for viability detection
5. ⏳ Evaluate performance metrics for live/dead differentiation

## Configuration
- **Sample Type**: Turkey rinsate from Cargill
- **Target Pathogen**: Salmonella (multiple serovars)
- **Cell Status**: Live cells vs. heat-killed (dead) cells
- **Heat Treatment**: Standardized heat-killing protocol

## Status
🚧 **Not Yet Implemented** - This notebook is a placeholder for future implementation.


In [ ]:
# Imports and Setup
# --------------------------------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import warnings

# Project-specific imports
from sensd_sers_analysis.data import load_dataset_by_serotypes, get_signals_matrix, get_raman_shift

warnings.filterwarnings("ignore")

# Plotting style
plt.style.use("default")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12

print("✅ All imports completed successfully")

## 1. Data Loading and Cell Status Detection

Load SERS data and identify live vs. dead cell samples based on filenames and metadata.

In [ ]:
# Load SERS Data and Identify Live vs Dead Samples
# --------------------------------------------------------------------------------------------------

# Define dataset location
data_folder = "../example_data/SERS Data 8 (April 2025)/"
signals_folders = ["March testing-Dilutions"]  # Adjust based on available data

# Load all serotypes to find both live and dead samples
serotypes = ["ST", "SE", "Mix"]

try:
    # Load data
    data_df = load_dataset_by_serotypes(data_folder, signals_folders, serotypes)
    print(f"✅ Successfully loaded {len(data_df)} data entries")

    if not data_df.empty:
        # Identify live vs dead samples based on filename keywords
        # Look for keywords indicating dead cells: "Dead", "Heat", "Killed", "Kill"
        dead_keywords = ["dead", "heat", "killed", "kill", "inactive"]
        live_keywords = ["live", "active", "viable"]

        def classify_cell_status(filename: str, serotype: str) -> str:
            """Classify cell status based on filename."""
            filename_lower = str(filename).lower()

            # Check for dead cell indicators
            for keyword in dead_keywords:
                if keyword in filename_lower:
                    return "Dead"

            # Check for live cell indicators
            for keyword in live_keywords:
                if keyword in filename_lower:
                    return "Live"

            # Default: assume live if no dead indicators found
            # (Most samples are likely live unless explicitly marked)
            return "Live"

        # Classify cell status
        data_df["cell_status"] = data_df.apply(
            lambda row: classify_cell_status(row["filename"], row["serotype"]), axis=1
        )

        print("\n📊 Cell Status Distribution:")
        print("=" * 60)
        status_counts = data_df["cell_status"].value_counts()
        for status, count in status_counts.items():
            print(f"  {status}: {count} entries")

        # Check distribution by serotype
        print("\nCell Status by Serotype:")
        for sero in serotypes:
            sero_data = data_df[data_df["serotype"] == sero]
            if not sero_data.empty:
                status_by_sero = sero_data["cell_status"].value_counts()
                print(f"\n  {sero}:")
                for status, count in status_by_sero.items():
                    print(f"    {status}: {count}")

        # Show sample filenames for each status
        print("\nSample filenames by status:")
        for status in ["Dead", "Live"]:
            status_data = data_df[data_df["cell_status"] == status]
            if not status_data.empty:
                sample_filenames = status_data["filename"].unique()[:5]
                print(f"\n  {status} samples:")
                for fn in sample_filenames:
                    print(f"    - {fn}")

except Exception as e:
    print(f"❌ Error loading data: {e}")
    import traceback

    traceback.print_exc()
    data_df = pd.DataFrame()

## 2. Filter to Comparable Samples

Filter to samples with similar serotype and concentration ranges for fair comparison.

In [ ]:
# Filter to Comparable Samples
# --------------------------------------------------------------------------------------------------

if not data_df.empty:
    # Check if we have both live and dead samples
    has_dead = "Dead" in data_df["cell_status"].values
    has_live = "Live" in data_df["cell_status"].values

    if has_dead and has_live:
        # Bin concentrations for fair comparison
        def bin_concentrations(concentrations: np.ndarray) -> np.ndarray:
            """Bin concentrations by order of magnitude."""
            log_concs = np.log10(concentrations + 1e-10)
            bins = np.floor(log_concs).astype(int)
            return bins

        data_df["concentration_bin"] = bin_concentrations(data_df["concentration"].values)

        # Find serotypes that have both live and dead samples
        comparison_groups = []
        for sero in serotypes:
            sero_data = data_df[data_df["serotype"] == sero]
            if len(sero_data) > 0:
                has_live_sero = "Live" in sero_data["cell_status"].values
                has_dead_sero = "Dead" in sero_data["cell_status"].values

                if has_live_sero and has_dead_sero:
                    # Find common concentration bins
                    live_bins = set(
                        sero_data[sero_data["cell_status"] == "Live"]["concentration_bin"].unique()
                    )
                    dead_bins = set(
                        sero_data[sero_data["cell_status"] == "Dead"]["concentration_bin"].unique()
                    )
                    common_bins = live_bins.intersection(dead_bins)

                    if len(common_bins) > 0:
                        comparison_groups.append(
                            {
                                "serotype": sero,
                                "common_bins": common_bins,
                                "live_count": len(sero_data[sero_data["cell_status"] == "Live"]),
                                "dead_count": len(sero_data[sero_data["cell_status"] == "Dead"]),
                            }
                        )

        print("\n📊 Comparable Sample Groups:")
        print("=" * 60)
        if comparison_groups:
            for group in comparison_groups:
                print(f"\n{group['serotype']}:")
                print(f"  Live samples: {group['live_count']}")
                print(f"  Dead samples: {group['dead_count']}")
                print(f"  Common concentration bins: {sorted(group['common_bins'])}")

            # Use the serotype with the most samples, or first available
            best_group = max(comparison_groups, key=lambda x: min(x["live_count"], x["dead_count"]))
            selected_sero = best_group["serotype"]
            selected_bins = best_group["common_bins"]

            # Filter to selected serotype and common bins
            filtered_df = data_df[
                (data_df["serotype"] == selected_sero)
                & (data_df["concentration_bin"].isin(selected_bins))
            ].copy()

            print(f"\n✅ Selected for analysis: {selected_sero} serotype")
            print(f"   Concentration bins: {sorted(selected_bins)}")
            print(f"   Total samples: {len(filtered_df)}")
            print(f"   Live: {len(filtered_df[filtered_df['cell_status'] == 'Live'])}")
            print(f"   Dead: {len(filtered_df[filtered_df['cell_status'] == 'Dead'])}")

            data_df = filtered_df
        else:
            print(
                "⚠️ No serotypes found with both live and dead samples in common concentration ranges"
            )
            print("   Proceeding with all available data...")
    else:
        print("\n⚠️ Missing cell status data:")
        print(f"   Has Dead samples: {has_dead}")
        print(f"   Has Live samples: {has_live}")
        if not has_dead:
            print(
                "   ⚠️ No dead cell samples found. Check filenames for keywords: dead, heat, killed"
            )
        if not has_live:
            print("   ⚠️ No live cell samples found.")

## 3. Average Spectra Comparison: Live vs Dead

Compare average spectra between live and dead cells.

In [ ]:
# Average Spectra Comparison: Live vs Dead
# --------------------------------------------------------------------------------------------------

if (
    not data_df.empty
    and "Dead" in data_df["cell_status"].values
    and "Live" in data_df["cell_status"].values
):
    raman_shift = get_raman_shift(data_df)

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    colors = {"Live": "#2ca02c", "Dead": "#d62728"}

    # Plot 1: Average spectra with standard deviation
    ax1 = axes[0]
    for status in ["Live", "Dead"]:
        status_data = data_df[data_df["cell_status"] == status]
        if len(status_data) > 0:
            signals_matrix = get_signals_matrix(status_data)
            mean_signal = np.mean(signals_matrix, axis=0)
            std_signal = np.std(signals_matrix, axis=0)

            ax1.plot(
                raman_shift,
                mean_signal,
                label=f"{status} (n={len(status_data)})",
                color=colors.get(status, "gray"),
                linewidth=2,
            )
            ax1.fill_between(
                raman_shift,
                mean_signal - std_signal,
                mean_signal + std_signal,
                alpha=0.2,
                color=colors.get(status, "gray"),
            )

    ax1.set_xlabel("Raman Shift (cm⁻¹)", fontsize=12)
    ax1.set_ylabel("Signal Intensity", fontsize=12)
    ax1.set_title("Average Spectra: Live vs Dead Cells", fontsize=14, fontweight="bold")
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # Plot 2: Difference spectrum (Live - Dead)
    ax2 = axes[1]
    live_data = data_df[data_df["cell_status"] == "Live"]
    dead_data = data_df[data_df["cell_status"] == "Dead"]

    if len(live_data) > 0 and len(dead_data) > 0:
        live_signals = get_signals_matrix(live_data)
        dead_signals = get_signals_matrix(dead_data)

        live_mean = np.mean(live_signals, axis=0)
        dead_mean = np.mean(dead_signals, axis=0)

        difference = live_mean - dead_mean

        ax2.plot(raman_shift, difference, label="Live - Dead", color="#1f77b4", linewidth=2)
        ax2.axhline(y=0, color="black", linestyle="--", linewidth=1, alpha=0.5)
        ax2.fill_between(
            raman_shift,
            0,
            difference,
            where=(difference >= 0),
            alpha=0.3,
            color="green",
            label="Live > Dead",
        )
        ax2.fill_between(
            raman_shift,
            0,
            difference,
            where=(difference < 0),
            alpha=0.3,
            color="red",
            label="Dead > Live",
        )

        ax2.set_xlabel("Raman Shift (cm⁻¹)", fontsize=12)
        ax2.set_ylabel("Difference in Signal Intensity", fontsize=12)
        ax2.set_title("Difference Spectrum: Live - Dead", fontsize=14, fontweight="bold")
        ax2.legend(fontsize=11)
        ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("✅ Average spectra comparison completed")
else:
    print("⚠️ Cannot compare: Need both Live and Dead samples")

## 4. Statistical Comparison and Feature Extraction

Identify spectral features that distinguish live from dead cells.

In [ ]:
# Statistical Comparison and Feature Extraction
# --------------------------------------------------------------------------------------------------

if (
    not data_df.empty
    and "Dead" in data_df["cell_status"].values
    and "Live" in data_df["cell_status"].values
):
    live_data = data_df[data_df["cell_status"] == "Live"]
    dead_data = data_df[data_df["cell_status"] == "Dead"]

    if len(live_data) > 0 and len(dead_data) > 0:
        raman_shift = get_raman_shift(data_df)
        live_signals = get_signals_matrix(live_data)
        dead_signals = get_signals_matrix(dead_data)

        # Calculate mean and std for each wavenumber
        live_mean = np.mean(live_signals, axis=0)
        dead_mean = np.mean(dead_signals, axis=0)
        live_std = np.std(live_signals, axis=0)
        dead_std = np.std(dead_signals, axis=0)

        # Perform t-test at each wavenumber
        p_values = []
        t_statistics = []
        for i in range(len(raman_shift)):
            t_stat, p_val = stats.ttest_ind(live_signals[:, i], dead_signals[:, i])
            t_statistics.append(t_stat)
            p_values.append(p_val)

        p_values = np.array(p_values)
        t_statistics = np.array(t_statistics)

        # Find significant differences (p < 0.05)
        significant_mask = p_values < 0.05
        significant_indices = np.where(significant_mask)[0]

        print("\n📊 Statistical Comparison Results:")
        print(f"Total wavenumber points: {len(raman_shift)}")
        print(
            f"Significantly different points (p < 0.05): {significant_mask.sum()} ({significant_mask.sum() / len(raman_shift) * 100:.1f}%)"
        )

        if len(significant_indices) > 0:
            print("\nTop 10 most significant differences:")
            top_indices = np.argsort(p_values)[:10]
            for idx in top_indices:
                print(
                    f"  {raman_shift[idx]:.1f} cm⁻¹: p={p_values[idx]:.2e}, t={t_statistics[idx]:.2f}"
                )

        # Visualization
        fig, axes = plt.subplots(2, 1, figsize=(14, 10))

        # Plot 1: Mean spectra with significance markers
        ax1 = axes[0]
        ax1.plot(raman_shift, live_mean, label="Live", color=colors["Live"], linewidth=2)
        ax1.fill_between(
            raman_shift, live_mean - live_std, live_mean + live_std, alpha=0.2, color=colors["Live"]
        )
        ax1.plot(raman_shift, dead_mean, label="Dead", color=colors["Dead"], linewidth=2)
        ax1.fill_between(
            raman_shift, dead_mean - dead_std, dead_mean + dead_std, alpha=0.2, color=colors["Dead"]
        )

        # Mark significant differences
        if len(significant_indices) > 0:
            ax1.scatter(
                raman_shift[significant_indices],
                np.maximum(live_mean[significant_indices], dead_mean[significant_indices]) * 1.1,
                marker="*",
                s=50,
                color="yellow",
                label="Significant (p<0.05)",
                zorder=5,
            )

        ax1.set_xlabel("Raman Shift (cm⁻¹)", fontsize=12)
        ax1.set_ylabel("Signal Intensity", fontsize=12)
        ax1.set_title(
            "Live vs Dead Spectra with Significance Markers", fontsize=14, fontweight="bold"
        )
        ax1.legend(fontsize=11)
        ax1.grid(True, alpha=0.3)

        # Plot 2: P-values and t-statistics
        ax2 = axes[1]
        ax2_twin = ax2.twinx()

        ax2.plot(
            raman_shift,
            -np.log10(p_values + 1e-10),
            color="blue",
            linewidth=1.5,
            label="-log10(p-value)",
        )
        ax2.axhline(
            y=-np.log10(0.05), color="red", linestyle="--", linewidth=1, label="p=0.05 threshold"
        )
        ax2.set_xlabel("Raman Shift (cm⁻¹)", fontsize=12)
        ax2.set_ylabel("-log10(p-value)", fontsize=12, color="blue")
        ax2.tick_params(axis="y", labelcolor="blue")
        ax2.legend(loc="upper left")

        ax2_twin.plot(
            raman_shift, t_statistics, color="green", linewidth=1.5, alpha=0.7, label="t-statistic"
        )
        ax2_twin.axhline(y=0, color="black", linestyle="-", linewidth=0.5, alpha=0.5)
        ax2_twin.set_ylabel("t-statistic", fontsize=12, color="green")
        ax2_twin.tick_params(axis="y", labelcolor="green")
        ax2_twin.legend(loc="upper right")

        ax2.set_title("Statistical Significance: Live vs Dead", fontsize=14, fontweight="bold")
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        print("✅ Statistical comparison completed")

## 5. Classification Model for Live/Dead Differentiation

Train a classification model to distinguish live from dead cells.

In [ ]:
# Classification Model for Live/Dead Differentiation
# --------------------------------------------------------------------------------------------------

if (
    not data_df.empty
    and "Dead" in data_df["cell_status"].values
    and "Live" in data_df["cell_status"].values
):
    live_data = data_df[data_df["cell_status"] == "Live"]
    dead_data = data_df[data_df["cell_status"] == "Dead"]

    if len(live_data) > 0 and len(dead_data) > 0 and len(data_df) >= 10:
        # Prepare data
        signals_matrix = get_signals_matrix(data_df)
        cell_status_labels = data_df["cell_status"].values

        print("\n📊 Classification Setup:")
        print(f"Live samples: {len(live_data)}")
        print(f"Dead samples: {len(dead_data)}")

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            signals_matrix,
            cell_status_labels,
            test_size=0.3,
            random_state=42,
            stratify=cell_status_labels,
        )

        # Train Random Forest classifier
        clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
        clf.fit(X_train, y_train)

        # Predictions
        y_pred = clf.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        # Results
        print("\n📊 Classification Results:")
        print(f"Accuracy: {accuracy * 100:.2f}%")
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred, target_names=["Dead", "Live"]))

        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred, labels=["Dead", "Live"])

        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Dead", "Live"],
            yticklabels=["Dead", "Live"],
            ax=ax,
        )
        ax.set_xlabel("Predicted", fontsize=12)
        ax.set_ylabel("Actual", fontsize=12)
        ax.set_title(
            f"Confusion Matrix: Live vs Dead (Accuracy: {accuracy * 100:.1f}%)",
            fontsize=14,
            fontweight="bold",
        )
        plt.tight_layout()
        plt.show()

        # Feature importance (top contributing wavenumbers)
        feature_importance = clf.feature_importances_
        top_indices = np.argsort(feature_importance)[-10:][::-1]

        print("\nTop 10 most important wavenumbers for classification:")
        for idx in top_indices:
            print(f"  {raman_shift[idx]:.1f} cm⁻¹: importance={feature_importance[idx]:.4f}")

        print("\n✅ Classification model completed")
    else:
        print("⚠️ Not enough samples for classification (need at least 10 total)")
else:
    print("⚠️ Cannot classify: Need both Live and Dead samples")